# 18 — Does model assistance save review time? (DR-58)

Review **20 new event types**, once each: 7 manual, 7 with Qwen3.6, and 6 with Gemini 3.8 Flash. The models use approved v1: title plus rubric, search off, up to two suggestions. Your final answer may contain more supported tags.

**Question:** do suggestions reduce active work and corrections? The primary measures are human time, correction burden, unresolved cases and context lookup. These separate small batches cannot establish a causal time saving or production accuracy. One person reviews everything, and topics differ between batches.

## How to participate

1. Run sections 1–3 with `NOTEBOOK18_RUN_LIVE=false` (the default). Prepared responses replay privately.
2. Open the rubric before beginning. Read the timing instructions in section 4.
3. Use **Start / resume** to reveal each event. Read it, choose final tags, and save. **Pause** excludes interruptions. Toggle **Looking up context** before and after researching a case; that time is included in active time.
4. Record whether you have seen the event before and whether it needs context. For unclear cases, say why. Sources are only the links you actually checked.
5. After all 20, run section 5. Do not repeat the same events as a fresh timed trial.

Private feed, suggestions and answers stay under `reports/practical-trial/v1/`. A public checkout cannot execute without those inputs. The public guide is `docs/plan/practical-tagging-trial.md`. Rerunning preserves saved answers. A kernel restart loses the unsaved current timer; record that interruption in the case notes.

In [ ]:
import json, sys, os, re, html, unicodedata, hashlib, difflib, copy
from pathlib import Path
from datetime import date, datetime
from collections import Counter
from statistics import median
from dotenv import load_dotenv
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'pyproject.toml').is_file())
sys.path[:0] = [str(root),str(root/'src')]
from notebooks.tools import verify_sha256, show_table, percent
from notebooks.tools.hashes import fingerprint
from notebooks.tools.hosted import chat_request
from notebooks.tools.gemini_evidence import request
from notebooks.tools.costs import BudgetGuard
from notebooks.tools.trial_review import show_trial_review, validate_review
from demo_logic.notebook_support import save_private_json
from demo_logic.taxonomy import topic_tags
from demo_logic.evaluation import multilabel_metrics
INPUTS = {'reports/practical-trial/v1/source-2026-10-06.json': 'a1c4603a01a3476bb80f546cd4b3ec1ec3f509ce6c178525b162911f90e51765', 'reports/practical-trial/v1/previous-title-registry.json': '5f1c479b88d971233f54545fca3b41c79d926ca648df5bb9c07489040e188a95', 'notebooks/prompts/practical-tagging-v1.json': '9b13b3125ab861118cad76cc6e90376fdb2eb3c536d343a12187b5e973538f8a', 'docs/tagging-rubric.md': '0127f65668271a6bc32ec6de32eb68c146228696ba6ecb69fbaf1a46cdba70bd'}
missing = [p for p in INPUTS if not (root/p).is_file()]
if missing: raise FileNotFoundError('Private trial inputs missing: '+', '.join(missing))
for p,h in INPUTS.items(): verify_sha256(root/p,h)
folder=root/'reports/practical-trial/v1'
source=json.loads((root/list(INPUTS)[0]).read_text())
registry=json.loads((root/list(INPUTS)[1]).read_text())
config=json.loads((root/list(INPUTS)[2]).read_text())
assert config['approval_status']=='approved' and config['output_policy']['maximum']==2
load_dotenv(root/'.env',override=False)
RUN_LIVE=os.getenv('NOTEBOOK18_RUN_LIVE','false').lower()=='true'
tags=sorted(topic_tags('v2'))


## 1. Select unfamiliar event types before seeing predictions

Use the 6 October public Berlin feed, for events dated 6 October–31 December 2026. Compare titles with the frozen registry of prior snapshot and experiment titles. For matching only, decode HTML, lowercase, remove parenthetical text and numbers, and collapse punctuation. Exclude exact or close matches: word-set overlap at least 0.55 or character similarity at least 0.78. Apply the same screen between selected events. This conservative screen can exclude distinct events and cannot prove different organizers or topics.

Use a fixed hash order to select 7 short titles (up to 8 words), 7 medium (9–20), and 6 long (21+). Rotate manual/Qwen/Gemini assignments within each length group and then interleave workflows. Titles determine the strata; labels and model results do not. Check prior familiarity during review and report familiar cases separately. This is a length-balanced convenience sample from currently published events, not a representative random sample of Berlin.

In [ ]:
def match_title(text):
    """Normalize only for overlap screening; preserve the original model input."""
    text=unicodedata.normalize('NFKC',html.unescape(text)).casefold()
    text=re.sub(r'\([^)]*\)',' ',text)
    return ' '.join(re.findall(r'[^\W\d_]+',text))

def similar(a,b):
    """Conservatively flag familiar wording, including changed dates/numbers."""
    x,y=set(a.split()),set(b.split())
    return a==b or (bool(x|y) and len(x&y)/len(x|y)>=.55) or difflib.SequenceMatcher(None,a,b).ratio()>=.78

def rank(text):
    """Fixed pseudorandom order, independent of labels and predictions."""
    return hashlib.sha256(('dr58-v1'+text).encode()).hexdigest()

known=sorted({match_title(t)for t in registry['titles']})
unique={}
for item in source['index']:
    try: event_date=datetime.strptime(item['datum'],'%d.%m.%Y').date()
    except (ValueError,KeyError): continue
    title=item.get('thema','').strip(); normalized=match_title(title)
    if not normalized or not date(2026,10,6)<=event_date<=date(2026,12,31):continue
    if normalized not in unique or (event_date,str(item['id'])) < (unique[normalized]['date'],str(unique[normalized]['source_id'])):
        unique[normalized]=dict(title=title,date=event_date,source_id=item['id'])
novel=[(k,v)for k,v in sorted(unique.items(),key=lambda kv:rank(kv[0])) if not any(similar(k,z)for z in known)]
selected=[];used=[];workflows=['manual','qwen','gemini']
for stratum,(low,high,target) in enumerate([(1,8,7),(9,20,7),(21,100000,6)]):
    batch=[]
    for k,v in novel:
        if not low<=len(v['title'].split())<=high or any(similar(k,z)for z in used):continue
        batch.append(dict(external_id=hashlib.sha256(('dr58:'+k).encode()).hexdigest()[:16],
            title=v['title'],date=v['date'].isoformat(),source_id=v['source_id'],
            normalized_title=k,stratum=['short','medium','long'][stratum],
            workflow=workflows[(len(batch)+stratum)%3]))
        used.append(k)
        if len(batch)==target:break
    if len(batch)!=target:raise ValueError('Insufficient novel titles in length stratum; revise sampling explicitly.')
    selected+=batch
queues={w:sorted([e for e in selected if e['workflow']==w],key=lambda e:rank(e['external_id']))for w in workflows}
events=[]
for turn in range(7):
    for w in workflows[turn%3:]+workflows[:turn%3]:
        if queues[w]:events.append(queues[w].pop(0))
assert len(events)==len({e['external_id']for e in events})==20
contract=dict(version='dr58-v1',date='2026-10-06',inputs=INPUTS,events=events,
    source_url='https://www.berlin.de/polizei/service/versammlungsbehoerde/versammlungen-aufzuege/index.php/index/all.json?q=',
    sampling='Title novelty screen; fixed hash order; length strata 7/7/6; rotated workflow assignment',
    counts=dict(Counter(e['workflow']for e in events)),raw_rows=len(source['index']),
    unique_date_eligible=len(unique),novel_candidates=len(novel),search=False,
    pricing_checked='2026-10-06',pricing_source='https://ai.google.dev/gemini-api/docs/pricing',
    gemini_input_output_usd_per_million=[.75,3.75],run_limit_usd=5,
    spending_observation={'reported_monthly_eur':2.10,'date':'2026-10-05'},
    limitation='Text novelty is not proven organizer novelty; one reviewer and unpaired topic batches.')
save_private_json(folder/'contract.json',contract)
show_table(['Workflow','Cases'],list(contract['counts'].items()))
print('Selection saved. Titles stay hidden until timed review.')


## 2. Prepare the assigned model suggestions

Only the 13 assisted cases are sent: 7 to Qwen3.6 and 6 to Gemini 3.8 Flash. Manual cases have no model call. Each request loads the approved v1 template exactly. Search is off, and no human answers are supplied. Google prices checked on 6 October match the existing adapter: $0.75/M input and $3.75/M output tokens. [Provider pricing](https://ai.google.dev/gemini-api/docs/pricing).

The shared budget guard reserves each Gemini attempt before sending. Errors pause preparation. Completed malformed answers are preserved as failures with no suggestions. The reviewer can still tag those events manually. System duration is measured from the original request; cached playback does not replace it. Suggestions are prepared in advance, so this trial measures review effort rather than live end-to-end waiting.

In [ ]:
guard=BudgetGuard(root/'reports/budget/capstone.sqlite',prior_total_microusd=0,prior_research_microusd=0,
    run_limits={'dr58-v1':5_000_000})
models={'qwen':config['candidates'][0],'gemini':config['candidates'][1]}
predictions={}
for e in events:
    if e['workflow']=='manual':continue
    c=models[e['workflow']];payload=copy.deepcopy(c['payload_template'])
    title_json=json.dumps({'title':e['title']},ensure_ascii=False)
    if e['workflow']=='qwen':
        payload['messages'][1]['content']=title_json
        result=chat_request({**c,'key_env':'HETZNER_API_KEY'},payload,tags=tags,
            cache_dir=folder/'qwen/cache',allow_live=RUN_LIVE,api_key=os.getenv('HETZNER_API_KEY'),max_retry_wait_seconds=30)
    else:
        payload['input']=title_json
        result=request(payload,event_id=e['external_id'],folder=folder/'gemini',guard=guard,
            key=os.getenv('GEMINI_API_KEY'),tags=tags,live=RUN_LIVE,run_id='dr58-v1',research_budget=False)
    returned=result.get('returned_model') or result.get('response',{}).get('model')
    if returned and returned!=c['model']:raise ValueError('Unexpected returned model; inspect privately before continuing.')
    malformed=result.get('status')=='invalid_response' or (result.get('error')=='Invalid or incomplete model response' and 'response'in result)
    if result['status']!='ok' and not malformed:raise RuntimeError(f"Preparation paused: {result['status']}. Successful responses are cached.")
    predictions[e['external_id']]={k:v for k,v in result.items()if k!='cached'}
    if malformed:assert not result['predicted']
    print(e['workflow'],len(predictions),'/13',result['status'],flush=True)
save_private_json(folder/'predictions.json',predictions)
print('Suggestions ready. Continue to the timed review; do not inspect raw predictions first.')


## 3. Check readiness

This table contains no event titles or suggested tags. All 20 events must be assigned exactly once. A failed response remains part of its assigned assisted workflow and is counted as a failure; it must not disappear from the analysis.

In [ ]:
assert set(predictions)=={e['external_id']for e in events if e['workflow']!='manual'}
show_table(['Check','Result'],[['Private cases',len(events)],['Prepared assisted cases',len(predictions)],
    ['Successful model responses',sum(p['status']=='ok'for p in predictions.values())],
    ['Model failures',sum(p['status']!='ok'for p in predictions.values())],
    ['Already saved human reviews',len(list((folder/'reviews').glob('*.json')))]])


## 4. Review one event at a time

Have the rubric ready. **Start / resume** reveals the title and starts timing. Read, decide and enter your answer while the timer runs. Select final tags with Cmd/Ctrl for multiple choices. Do not time breaks: use **Pause**. Press **Save and next** when finished; the next title stays hidden until Start.

Use **Looking up context** for time spent searching or checking sources, then switch it off. Add URLs you actually read and a short note about what was missing. If no evidence resolves the topic, select **Unclear**. For no fitting tag, select **Resolved**, leave tags empty and explain why. A familiar event stays in the trial but is reported separately.

Do not open source data or raw model predictions beforehand. Model identity is visible, so expectations may influence review. Final answers made after seeing suggestions are assisted decisions; they are not independent truth labels.

In [ ]:
review_panel=show_trial_review(events,predictions,folder/'reviews',tags,fingerprint(contract))

## 5. Summarize after the human work

Until every case is saved, show progress only. Report active time, its context subset, corrections, unresolved answers, prior exposure, model failures, coverage and original latency. Median times come from different event batches; they are descriptive, not a controlled estimate of time saved.

For assisted cases, tag agreement with final answers is influenced by seeing suggestions. Any precision/recall/F1 below is labeled **assisted agreement** and cannot replace independent benchmark quality. Unclear cases are excluded from agreement scoring and counted explicitly. Manual tagging has no model quality score.

In [ ]:
reviews={}
for e in events:
    p=folder/'reviews'/(e['external_id']+'.json')
    if p.exists():
        answer=json.loads(p.read_text());validate_review(answer,tags)
        assert answer['contract_sha256']==fingerprint(contract) and answer['external_id']==e['external_id']
        expected=fingerprint(predictions[e['external_id']])if e['external_id']in predictions else None
        assert answer['prediction_sha256']==expected
        reviews[e['external_id']]=answer
print(f'{len(reviews)}/20 reviews saved')
if len(reviews)==20:
    summary={}
    for w in workflows:
        items=[reviews[e['external_id']]for e in events if e['workflow']==w]
        resolved=[x for x in items if x['status']=='resolved']
        unfamiliar=[x for x in items if x['prior_exposure']=='no']
        result=dict(cases=len(items),resolved=len(resolved),unclear=len(items)-len(resolved),
            median_active_seconds=median(x['active_seconds']for x in items),
            median_context_seconds=median(x['context_seconds']for x in items),
            context_needed=sum(x['context_needed']for x in items),unfamiliar=len(unfamiliar),
            unfamiliar_median_active_seconds=median(x['active_seconds']for x in unfamiliar)if unfamiliar else None)
        if w!='manual':
            ps=[predictions[x['external_id']]for x in items]
            successful_resolved=[x for x in resolved if predictions[x['external_id']]['status']=='ok']
            result.update(failures=sum(p['status']!='ok'for p in ps),
                coverage=sum(bool(p['predicted'])for p in ps)/len(ps),
                accepted_unchanged=sum(set(x['final_tags'])==set(predictions[x['external_id']]['predicted'])for x in successful_resolved),
                resolved_successes=len(successful_resolved),
                tags_added=sum(len(set(x['final_tags'])-set(predictions[x['external_id']]['predicted']))for x in resolved),
                tags_removed=sum(len(set(predictions[x['external_id']]['predicted'])-set(x['final_tags']))for x in resolved),
                median_original_request_seconds=median(p['duration_seconds']for p in ps),
                over_five_minutes=sum(p['duration_seconds']>=300 for p in ps),
                upper_usd_estimate=sum(p.get('gross_upper_microusd',0)or p.get('maximum_microusd',0)for p in ps)/1e6)
            if resolved:result['assisted_agreement']=multilabel_metrics([x['final_tags']for x in resolved],[predictions[x['external_id']]['predicted']for x in resolved],tags)
        summary[w]=result
    save_private_json(folder/'human-results-v1.json',dict(contract_sha256=fingerprint(contract),summary=summary,
        review_hashes={i:fingerprint(x)for i,x in reviews.items()},limitation='Assisted answers are not independent reference labels.'))
    show_table(['Workflow','Cases','Median active seconds','Median context seconds','Unclear','Unfamiliar'],
        [[w,s['cases'],round(s['median_active_seconds'],1),round(s['median_context_seconds'],1),s['unclear'],s['unfamiliar']]for w,s in summary.items()])
    show_table(['Workflow','Accepted unchanged / resolved successful','Tags added','Tags removed','Failures','Coverage'],
        [[w,f"{s['accepted_unchanged']}/{s['resolved_successes']}",s['tags_added'],s['tags_removed'],s['failures'],percent(s['coverage'])]for w,s in summary.items()if w!='manual'])
    show_table(['Workflow','Assisted precision','Assisted recall','Assisted F1'],
        [[w,*[percent(s['assisted_agreement'][k]) for k in ['micro_precision','micro_recall','micro_f1']]]for w,s in summary.items()if 'assisted_agreement'in s])
else:
    print('Complete the remaining reviews in section 4 before comparing workflows.')


## 6. Conclusion — awaiting human timing

Preparation does not tell us which workflow saves effort. After the 20 reviews, compare active time and correction burden, inspect prior exposure and context needs, and record whether the evidence supports proceeding to DR-27. Keep the independent 80% precision, 80% recall and 95% coverage targets alongside this small practical study. A faster workflow alone does not establish acceptable quality.